In [5]:
###########################################################
# Mixed-Precision Block Matrix Multiply (Algorithm 4.1)
#   - Input A, B in Float32
#   - Internally cast to Float16 (low precision) for speed
#   - Accumulate each block product in Float32 (high precision)
#   - Final result is stored in Float16
###########################################################

using LinearAlgebra
using BenchmarkTools

function mixed_precision_block_gemm(A::Matrix{Float32},
                                    B::Matrix{Float32},
                                    bsize::Int)
    m, n = size(A)
    n2, t = size(B)
    @assert n == n2 "Incompatible matrix sizes"

    # Convert A and B to low-precision (simulating f_low storage)
    A_low = Float16.(A)
    B_low = Float16.(B)

    # Number of blocks along each dimension
    p = cld(m, bsize)
    q = cld(n, bsize)
    r = cld(t, bsize)

    C = zeros(Float16, m, t)

    for i = 1:p
        ib = ((i - 1) * bsize + 1) : min(i * bsize, m)
        for j = 1:r
            jb = ((j - 1) * bsize + 1) : min(j * bsize, t)

            # Accumulator in f_high = Float32
            Cij_high = zeros(Float32, length(ib), length(jb))

            for l = 1:q
                lb = ((l - 1) * bsize + 1) : min(l * bsize, n)

                # Promote each block to Float32 for the block-FMA
                A_block = Float32.(A_low[ib, lb])
                B_block = Float32.(B_low[lb, jb])

                # Block-FMA: Cij_high = Cij_high + A_block * B_block
                mul!(Cij_high, A_block, B_block, 1f0, 1f0)
            end

            # Convert accumulator back to low precision
            C[ib, jb] .= Float16.(Cij_high)
        end
    end
    return C
end

###########################################################
# Testing wrapper
###########################################################
function test_kernel(m, n, t, bsize=32)
    println("Test: $m×$n * $n×$t, block size = $bsize")

    # Generate matrices in Float32 (f_high input format)
    A32 = randn(Float32, m, n)
    B32 = randn(Float32, n, t)

    # Reference result in Float64
    C_true = Float64.(A32) * Float64.(B32)

    # Time the mixed-precision multiply
    t_run = @belapsed mixed_precision_block_gemm($A32, $B32, $bsize)
    C_mp = mixed_precision_block_gemm(A32, B32, bsize)

    # Compute absolute error vs reference
    C_mp64 = Float64.(C_mp)
    err = maximum(abs.(C_true .- C_mp64))

    println("Runtime (sec): ", t_run)
    println("Max absolute error: ", err)
end

# Example run
test_kernel(256, 256, 256, 32)


Matrix sizes: 256×256 times 256×256, block size = 32
Runtime (sec): 0.00097125
Max absolute error: 0.030549047991627276


In [19]:
using LinearAlgebra

# Mixed precision iterative refinement solver for Ax = b
function mixed_precision_iterative_refinement(A::Matrix{Float32}, b::Vector{Float32}; max_iters=5)
    n = size(A, 1)
    
    # Convert A and b to low precision (Float16) for approximate solve
    A_low = Float16.(A)
    b_low = Float16.(b)

    # Initial solve in low precision
    x_low = A_low \ b_low  # roughly solve in Float16
    
    # Promote to Float32 for refinement
    x = Float32.(x_low)
    
    for iter in 1:max_iters
        # Compute residual r = b - A*x in higher precision
        r = b - A * x
        
        # Check convergence
        res_norm = norm(r)
        println("Iteration $iter, residual norm = $res_norm")
        if res_norm < 1e-6
            break
        end
        
        # Solve correction term A * delta = r in low precision
        r_low = Float16.(r)
        delta_low = A_low \ r_low
        
        # Update solution in Float32
        delta = Float32.(delta_low)
        x += delta
    end
    
    return x
end

# Example usage
n = 10
A = rand(Float32, n, n)
b = rand(Float32, n)

x_refined = mixed_precision_iterative_refinement(A, b)
println("Refined solution x = ", x_refined)


Iteration 1, residual norm = 0.0024480268
Iteration 2, residual norm = 4.4771696e-5
Iteration 3, residual norm = 8.095288e-7
Refined solution x = Float32[0.2985102, 0.034644604, 0.9020207, 1.0609064, -2.0343795, 0.38416684, 1.748405, -0.22335339, -0.7948274, 0.031692386]
